## DATA2-208

In [0]:
env="p"

In [0]:
spark.sql(f"""
UPDATE `{env}-cdh`.primaryactive.portfolio
SET AssetOwner="Waterweal" 
 WHERE PortfolioName in ("MONZA PORTFOLIO - RETAIL","MONZA PORTFOLIO - WHOLESALE","MONZA PORTFOLIO - D")
""")


spark.sql(f"""
UPDATE `{env}-cdh`.primaryactive.portfolio
SET Short_Name_Asset_Owner="Waterweal"
 WHERE PortfolioName in ("MONZA PORTFOLIO - RETAIL","MONZA PORTFOLIO - WHOLESALE","MONZA PORTFOLIO - D")
""")

In [0]:
spark.sql(f'''
WITH RankedRows AS (
        SELECT 
            ROW_NUMBER() OVER (PARTITION BY BusinessDate, ExposureDerivativeID ORDER BY LastModificationDate desc) AS row_num,
            BusinessDate, ExposureDerivativeID, LastModificationDate
        FROM `{env}-cdh`.derivativearchive.dexposure
        WHERE BusinessDate = '2022-04-29' 
)
DELETE FROM `{env}-cdh`.derivativearchive.dexposure dexp
WHERE EXISTS (
    SELECT 1
    FROM RankedRows r
    WHERE r.row_num = 2
      AND dexp.BusinessDate = r.BusinessDate
      AND dexp.ExposureDerivativeID = r.ExposureDerivativeID
      AND dexp.LastModificationDate is null
);
''')

## HighBalance field update in Dparty

In [0]:
spark.sql(f"""
MERGE INTO `{env}-cdh`.derivativeactive.DParty dp
  USING (
    SELECT distinct CDI, BusinessDate
    FROM `{env}-cdh`.primary.exposure
    WHERE PartyRelationshipType = 'Primary Owner'
    AND LegalActionPriority = 'HAPS_High_Balance'
    AND SourceSystem = 'qca'
   ) hb
   ON hb.CDI = dp.CDI
   AND hb.BusinessDate = dp.BusinessDate
  WHEN MATCHED THEN
  UPDATE SET
    dp.HighBalanceFlag = case when hb.CDI is not null then true else false end ;
""")

In [0]:
spark.sql(f"""
UPDATE `{env}-cdh`.derivativearchive.dexposure
SET PartyRelationshipType = 'Primary Owner'
WHERE PartyRelationshipType='MPrimary Owner';
""")